# Первая практика — мини-анализ HMDA

## Подготовка

In [ ]:
from pathlib import Path
import sys
import pandas as pd

print("Python:", sys.executable)
print("pandas:", pd.__version__)
print("Рабочая папка:", Path.cwd())
DATA = Path("../data")
assert DATA.is_dir(), "Рядом с папкой ноутбука должна находиться data/"

## Маленький пример reading_example.csv

In [ ]:
example = pd.read_csv(
    DATA / "reading_example.csv",
    dtype={"lei": "string", "county_code": "string"},
)
example

## A. Разделитель CSV

In [ ]:
wrong_sep = pd.read_csv(DATA / "reading_example.csv", sep=";")
print(wrong_sep.shape)
wrong_sep

## B. Частичное чтение

In [ ]:
preview = pd.read_csv(DATA / "reading_example.csv", nrows=3)
small_columns = pd.read_csv(
    DATA / "reading_example.csv", usecols=["action_taken", "income"]
)
print(preview.shape)
print(small_columns.shape)

## Выбор Series и DataFrame

In [ ]:
outcomes = example["action_taken"]
print(type(outcomes))
outcomes

## Выбор нескольких столбцов

In [ ]:
names = ["source_row", "action_taken", "income"]
selected = example[names]
print(type(selected))
selected.head(6)

## C. Выбор столбцов и просмотр

In [ ]:
amount_view = example[["source_row", "loan_amount"]].head(2)
print(amount_view)
print(example.shape)
print(example.columns)
example.info()

## Пример расчёта одобрений

In [ ]:
approved_flag = outcomes.isin([1, 2])
approved_flag

## Количество и доля одобрений

In [ ]:
n_approved = approved_flag.sum()
n_decisions = len(example)
assert n_decisions > 0
small_approval_rate = n_approved / n_decisions
print(n_approved, n_decisions)
print(f"Одобрено: {100 * small_approval_rate:.2f}%")

## D. Доля выдач на маленьком примере

In [ ]:
issued_flag = outcomes.eq(1)
n_issued = issued_flag.sum()
small_origination_rate = n_issued / n_decisions
print(issued_flag)
print(n_issued, n_decisions)
print(f"Выдано: {100 * small_origination_rate:.2f}%")

## Основной набор decisions.csv

In [ ]:
decisions = pd.read_csv(
    DATA / "decisions.csv", dtype={"lei": "string", "county_code": "string"}
)
full_outcomes = decisions["action_taken"]
full_n = len(decisions)
print(decisions.shape)
full_outcomes.value_counts(dropna=False)

## E. Две доли на основном наборе

In [ ]:
full_approved = full_outcomes.isin([1, 2]).sum()
full_issued = full_outcomes.eq(1).sum()
full_approval_rate = full_approved / full_n
full_origination_rate = full_issued / full_n
print(f"Одобрено {full_approved}/{full_n}: {100 * full_approval_rate:.2f}%")
print(f"Выдано {full_issued}/{full_n}: {100 * full_origination_rate:.2f}%")

## Проверка входных данных

In [ ]:
assert full_n > 0
assert full_outcomes.isna().sum() == 0
assert full_outcomes.isin([1, 2, 3]).sum() == full_n
income_missing = decisions["income"].isna().sum()
print("Пропущен доход:", income_missing)

## F. Баланс и диапазоны

In [ ]:
full_denied = full_outcomes.eq(3).sum()
assert full_approved + full_denied == full_n
assert 0 <= full_issued <= full_approved <= full_n
assert 0 <= full_origination_rate <= full_approval_rate <= 1
print("Отказы:", full_denied)

## G. Расчёт из отчёта

In [ ]:
approval_rate = full_outcomes.eq(1).sum() / full_n
approval_rate

## G. Исправление условия

In [ ]:
correct_approval_rate = full_outcomes.isin([1, 2]).sum() / full_n
print(f"Одобрено: {100 * correct_approval_rate:.2f}%")

## G. Исправление названия

In [ ]:
origination_rate = full_outcomes.eq(1).sum() / full_n
print(f"Выдано: {100 * origination_rate:.2f}%")